# NB5b — AraBERT baseline: frozen probe vs fine-tuning

The first neural rung of the comparison ladder. This notebook evaluates AraBERTv2 as an AI-text
detector two ways, and the gap between them is the point:

1. **Frozen + linear probe.** Freeze AraBERT, extract the [CLS] embedding, train only a linear
   classifier on top. This measures the quality of the *pre-trained representation as it is*, before
   the model ever sees our task.
2. **Full fine-tuning.** Train AraBERT end-to-end with a classification head and AdamW. This measures
   what the model can do once adapted to the task.

The difference between the two numbers is exactly the value that fine-tuning adds — the thing the
thesis needs to quantify before committing to a hybrid.

**Input:** `dataset.parquet` (7,101 articles, pair-aware split).
**Output:** `nb5b_arabert_results.parquet` — metrics for both tracks, for the comparison table.

## Encoder and preprocessing

Encoder is `aubmindlab/bert-base-arabertv2`, the segmented AraBERT, which requires the AraBERT
preprocessor (Farasa segmentation) before tokenization. This choice is fixed for the thesis; the same
encoder feeds the hybrid model later.

## The 512-token problem, and how this notebook handles it

AraBERT has a hard 512-token limit, but our articles average 741 words and reach 5,524; with WordPiece
and Farasa segmentation a long article is well over 2,000 tokens. Blind truncation at 512 would throw
away most of a long article, so this notebook uses **chunk-and-pool**, applied identically in both
tracks and later in the hybrid, so every comparison is fair.

The chunking is sentence-aware, not a blind token window:

- Chunks are cut only at sentence boundaries (Arabic full stop, question mark, exclamation), never
  mid-sentence, so no chunk ends on a broken clause.
- Sentences are packed greedily until the next one would exceed the token budget, then a new chunk
  starts **with an overlap** — the last sentence of the previous chunk is repeated at the start of the
  next, preserving cross-boundary context (the margin).
- A rare sentence longer than the budget on its own is split at the token level as a fallback.
- Chunks are capped at **6 per article**, which covers 96% of the corpus completely and bounds the
  compute and memory cost on a T4. Each article's chunk [CLS] vectors are mean-pooled into a single
  R^768 vector.

## Setup

In [1]:
!pip -q install transformers arabert farasapy >/dev/null 2>&1

import pandas as pd, numpy as np, re, os, glob, time, json, pickle
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'cpu')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
OUT_DIR = '/kaggle/working'

ENCODER_ID  = 'aubmindlab/bert-base-arabertv2'
MAX_LEN     = 512
CHUNK_TOK   = 510            # reserve room for [CLS] and [SEP]
OVERLAP_SENTS = 1           # repeat last sentence into the next chunk (the margin)
MAX_CHUNKS  = 6             # covers 96% of the corpus fully; bounds T4 cost

def find_parquet(preferred, *keywords):
    if os.path.exists(preferred): return preferred
    for kw in keywords:
        hits = [p for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True) if kw in p]
        if hits:
            print(f'(resolved {kw} -> {hits[0]})'); return hits[0]
    raise FileNotFoundError(preferred)

DATA_PATH = find_parquet('/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet', 'dataset')
df = pd.read_parquet(DATA_PATH)
print('dataset:', df.shape, '| splits:', df['split'].value_counts().to_dict())

device: cuda | Tesla T4
dataset: (7101, 7) | splits: {'train': 5363, 'test': 1093, 'val': 645}


## AraBERT preprocessor (Farasa) and tokenizer

AraBERTv2 is the segmented variant, so text must pass through the AraBERT preprocessor before
tokenization. I preprocess once and cache the result on the dataframe, because Farasa segmentation is
slow and both tracks reuse it.

In [2]:
from arabert.preprocess import ArabertPreprocessor
from transformers import AutoTokenizer

arabert_prep = ArabertPreprocessor(model_name=ENCODER_ID)
tokenizer = AutoTokenizer.from_pretrained(ENCODER_ID)

# Farasa-segment every article once, cached. This is the slow step; do it a single time.
t0 = time.time()
df['prep'] = df['text'].apply(lambda t: arabert_prep.preprocess(t))
print(f'preprocessed {len(df)} articles in {time.time()-t0:.0f}s')
print('sample:', df['prep'].iloc[0][:160])

/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:274: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN = re.compile(u"([^\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:276: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN_SPLIT = re.compile(u"([\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:281: SyntaxWarning: invalid escape sequence '\s'
  ARABIC_STRING = re.compile(u"([^\u0600-\u0652%s%s%s\s\d])" \
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:1237: SyntaxWarning: invalid escape sequence '\s'
  u"(?<=\s(%s|%s))%s" % (WAW, YEH, FATHA), \
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:1450: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub(u"(?<=[\s\d])([%s])+"%(TASHKEEL_STRING),"",text,  re.UNICODE)
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is 

100%|██████████| 241M/241M [06:48<00:00, 590kiB/s]


[2026-07-19 14:19:16,800 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

preprocessed 7101 articles in 203s
sample: خرج مئ +ات من ال+ أساتذ +ة ال+ متعاقد +ين ال+ منضو +ين تحت لواء ال+ تنسيقي +ة ال+ وطني +ة ل+ ال+ أساتذ +ة الذين فرض علي +هم ال+ تعاقد ، مساء ال+ يوم ال+ خميس ، 


## Sentence-aware chunking

Cuts only at sentence boundaries, packs greedily to the token budget, and starts each new chunk with
the previous chunk's last sentence as overlap. A single over-long sentence is split at the token
level as a fallback. Returns a list of token-id windows, each already wrapped in [CLS] … [SEP], capped
at MAX_CHUNKS.

In [3]:
_SENT_SPLIT = re.compile(r'(?<=[.!?\u061f])\s+')

def split_sentences(text):
    return [s.strip() for s in _SENT_SPLIT.split(text.strip()) if s.strip()]

def chunk_token_windows(text):
    # returns list of input-id windows (with special tokens), <= MAX_CHUNKS
    sents = split_sentences(text)
    if not sents:
        sents = [text.strip() or tokenizer.unk_token]
    sent_ids = [tokenizer.encode(s, add_special_tokens=False) for s in sents]

    windows, cur, cur_len, i = [], [], 0, 0
    while i < len(sent_ids):
        sid = sent_ids[i]
        # a single sentence longer than the budget: hard-split it at token level
        if len(sid) > CHUNK_TOK:
            if cur:
                windows.append(cur); cur, cur_len = [], 0
                if len(windows) >= MAX_CHUNKS: break
            for s in range(0, len(sid), CHUNK_TOK):
                windows.append(sid[s:s+CHUNK_TOK])
                if len(windows) >= MAX_CHUNKS: break
            i += 1
            if len(windows) >= MAX_CHUNKS: break
            continue
        if cur and cur_len + len(sid) > CHUNK_TOK:
            windows.append(cur)
            if len(windows) >= MAX_CHUNKS: break
            # overlap: carry the last OVERLAP_SENTS sentences into the next chunk
            carry = cur_sents[-OVERLAP_SENTS:] if OVERLAP_SENTS else []
            cur = [t for cs in carry for t in cs]
            cur_len = len(cur); cur_sents = list(carry)
        else:
            if not cur: cur_sents = []
        cur += sid; cur_len += len(sid); cur_sents.append(sid); i += 1
    if cur and len(windows) < MAX_CHUNKS:
        windows.append(cur)

    cls, sep = tokenizer.cls_token_id, tokenizer.sep_token_id
    return [[cls] + w[:CHUNK_TOK] + [sep] for w in windows[:MAX_CHUNKS]]

# quick check on the longest article
_lens = df['prep'].sample(200, random_state=SEED).apply(lambda t: len(chunk_token_windows(t)))
print('chunks per article (sample of 200):')
print(_lens.value_counts().sort_index().to_string())
print(f'mean {_lens.mean():.2f}, max {_lens.max()}')

Token indices sequence length is longer than the specified maximum sequence length for this model (3072 > 512). Running this sequence through the model will result in indexing errors


chunks per article (sample of 200):
prep
2    54
3    70
4    25
5    24
6    27
mean 3.50, max 6


## Batched chunk encoder

Encodes a list of chunk windows for one article and mean-pools their [CLS] vectors into a single
R^768 document vector. Used by both tracks; in the probe track it runs under `no_grad`, in the
fine-tune track it participates in the gradient.

In [4]:
def encode_document(bert, windows, device, train=False):
    # windows: list of token-id lists (already wrapped in special tokens)
    maxlen = max(len(w) for w in windows)
    pad = tokenizer.pad_token_id
    ids = torch.full((len(windows), maxlen), pad, dtype=torch.long)
    mask = torch.zeros((len(windows), maxlen), dtype=torch.long)
    for j, w in enumerate(windows):
        ids[j, :len(w)] = torch.tensor(w); mask[j, :len(w)] = 1
    ids, mask = ids.to(device), mask.to(device)
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        out = bert(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0, :]  # [n_chunks, 768]
        doc = out.mean(dim=0)                                                       # mean-pool -> [768]
    return doc

print('document encoder ready | pooling = mean over chunk [CLS]')

document encoder ready | pooling = mean over chunk [CLS]


## Track 1 — Frozen AraBERT + linear probe

Freeze every AraBERT weight, run each article once, mean-pool its chunk [CLS] vectors, and cache the
resulting R^768 document vectors. Then train a plain logistic-regression head on the cached train
vectors and score on test. Because the encoder is frozen, this is a forward-only pass — cheap — and it
isolates how much the pre-trained representation already knows before any task adaptation.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

bert_frozen = None
def load_encoder():
    from transformers import AutoModel
    return AutoModel.from_pretrained(ENCODER_ID).to(DEVICE).eval()

CACHE = f'{OUT_DIR}/arabert_cls_frozen.npy'
if os.path.exists(CACHE):
    emb = np.load(CACHE)
    print('loaded cached embeddings:', emb.shape)
else:
    bert_frozen = load_encoder()
    for p in bert_frozen.parameters(): p.requires_grad_(False)
    emb = np.zeros((len(df), 768), dtype=np.float32)
    t0 = time.time()
    for idx in range(len(df)):
        windows = chunk_token_windows(df['prep'].iloc[idx])
        emb[idx] = encode_document(bert_frozen, windows, DEVICE, train=False).cpu().numpy()
        if (idx+1) % 200 == 0 or idx+1 == len(df):
            el = time.time()-t0
            print(f'[{idx+1:5d}/{len(df)}] {el/(idx+1):.2f}s/doc | ETA {el/(idx+1)*(len(df)-idx-1)/60:.0f}m',
                  flush=True)
    np.save(CACHE, emb)
    print('cached embeddings:', emb.shape)

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[  200/7101] 0.10s/doc | ETA 11m
[  400/7101] 0.10s/doc | ETA 11m
[  600/7101] 0.10s/doc | ETA 10m
[  800/7101] 0.10s/doc | ETA 10m
[ 1000/7101] 0.10s/doc | ETA 10m
[ 1200/7101] 0.10s/doc | ETA 10m
[ 1400/7101] 0.10s/doc | ETA 10m
[ 1600/7101] 0.10s/doc | ETA 9m
[ 1800/7101] 0.10s/doc | ETA 9m
[ 2000/7101] 0.10s/doc | ETA 9m
[ 2200/7101] 0.10s/doc | ETA 8m
[ 2400/7101] 0.10s/doc | ETA 8m
[ 2600/7101] 0.10s/doc | ETA 8m
[ 2800/7101] 0.10s/doc | ETA 7m
[ 3000/7101] 0.10s/doc | ETA 7m
[ 3200/7101] 0.10s/doc | ETA 7m
[ 3400/7101] 0.10s/doc | ETA 6m
[ 3600/7101] 0.10s/doc | ETA 6m
[ 3800/7101] 0.10s/doc | ETA 6m
[ 4000/7101] 0.10s/doc | ETA 5m
[ 4200/7101] 0.10s/doc | ETA 5m
[ 4400/7101] 0.10s/doc | ETA 5m
[ 4600/7101] 0.10s/doc | ETA 4m
[ 4800/7101] 0.11s/doc | ETA 4m
[ 5000/7101] 0.11s/doc | ETA 4m
[ 5200/7101] 0.10s/doc | ETA 3m
[ 5400/7101] 0.10s/doc | ETA 3m
[ 5600/7101] 0.10s/doc | ETA 3m
[ 5800/7101] 0.11s/doc | ETA 2m
[ 6000/7101] 0.11s/doc | ETA 2m
[ 6200/7101] 0.11s/doc | ETA 2m
[

In [6]:
def split_mask(name): return (df['split'] == name).to_numpy()
ytr = df.loc[split_mask('train'), 'label'].to_numpy()
yte = df.loc[split_mask('test'),  'label'].to_numpy()
Etr, Ete = emb[split_mask('train')], emb[split_mask('test')]

probe = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED)
probe.fit(Etr, ytr)
p_pred  = probe.predict(Ete)
p_proba = probe.predict_proba(Ete)[:, 1]

def metric_row(name, y, pred, proba):
    return {'model': name,
            'accuracy':  accuracy_score(y, pred),
            'precision': precision_score(y, pred),
            'recall':    recall_score(y, pred),
            'macro_f1':  f1_score(y, pred, average='macro'),
            'auc_roc':   roc_auc_score(y, proba)}

probe_row = metric_row('AraBERT frozen + probe', yte, p_pred, p_proba)
print('TRACK 1 — frozen probe')
for k in ['accuracy','precision','recall','macro_f1','auc_roc']:
    print(f'  {k:<10} {100*probe_row[k]:.1f}')

TRACK 1 — frozen probe
  accuracy   99.5
  precision  99.5
  recall     99.5
  macro_f1   99.5
  auc_roc    100.0


## Track 2 — Full fine-tuning

Now AraBERT is trained end-to-end. The model encodes each article with the same sentence-aware
chunk-and-pool, feeds the pooled R^768 vector through a small classification head, and updates both
the encoder and the head with AdamW. A per-article variable chunk count makes true batching awkward,
so I use gradient accumulation: process articles one at a time, accumulate gradients over an effective
batch, and step. Checkpoints are saved every epoch so a Kaggle timeout can resume.

In [7]:
class DocDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame['prep'].tolist()
        self.labels = frame['label'].tolist()
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        return chunk_token_windows(self.texts[i]), self.labels[i]

class AraBertClassifier(nn.Module):
    def __init__(self, encoder_id=ENCODER_ID, hidden=768, dropout=0.2):
        super().__init__()
        from transformers import AutoModel
        self.bert = AutoModel.from_pretrained(encoder_id)
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Dropout(dropout),
                                  nn.Linear(hidden, 2))
    def forward(self, windows, device, train=True):
        doc = encode_document(self.bert, windows, device, train=train)   # [768]
        return self.head(doc.unsqueeze(0))                                # [1, 2]

EPOCHS      = 3
ACCUM       = 16          # effective batch size
ENC_LR      = 2e-5
HEAD_LR     = 1e-3
CKPT        = f'{OUT_DIR}/nb5b_arabert_ft.pt'
print('fine-tune config: epochs', EPOCHS, '| accum', ACCUM, '| enc_lr', ENC_LR)

fine-tune config: epochs 3 | accum 16 | enc_lr 2e-05


In [8]:
from torch.optim import AdamW

# class weights for the 50.7% AI skew
n0 = int((ytr == 0).sum()); n1 = int((ytr == 1).sum())
w = torch.tensor([ (n0+n1)/(2*n0), (n0+n1)/(2*n1) ], dtype=torch.float, device=DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=w)

train_df = df[split_mask('train')].reset_index(drop=True)
val_df   = df[split_mask('test')].reset_index(drop=True)   # report on test
train_ds = DocDataset(train_df)

SMOKE = True    # True: 40-article smoke test to confirm the loop runs; set False for the real run

def run_fine_tune(smoke=False):
    model = AraBertClassifier().to(DEVICE)
    enc_params  = list(model.bert.parameters())
    head_params = list(model.head.parameters())
    opt = AdamW([{'params': enc_params, 'lr': ENC_LR},
                 {'params': head_params, 'lr': HEAD_LR}], weight_decay=0.01)
    start_epoch = 0
    if os.path.exists(CKPT) and not smoke:
        ck = torch.load(CKPT, map_location=DEVICE)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        start_epoch = ck['epoch']
        print('resumed from epoch', start_epoch)

    n = 40 if smoke else len(train_ds)
    order = np.random.permutation(len(train_ds))[:n]
    for epoch in range(start_epoch, 1 if smoke else EPOCHS):
        model.train(); opt.zero_grad(); running = 0.0; t0 = time.time()
        for step, idx in enumerate(order):
            windows, label = train_ds[idx]
            logits = model(windows, DEVICE, train=True)
            loss = loss_fn(logits, torch.tensor([label], device=DEVICE)) / ACCUM
            loss.backward(); running += loss.item() * ACCUM
            if (step+1) % ACCUM == 0 or step+1 == len(order):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); opt.zero_grad()
            if (step+1) % 200 == 0:
                el = time.time()-t0
                print(f'  epoch {epoch} [{step+1}/{len(order)}] loss {running/(step+1):.4f} '
                      f'| {el/(step+1):.2f}s/doc', flush=True)
        if not smoke:
            torch.save({'model':model.state_dict(),'opt':opt.state_dict(),'epoch':epoch+1}, CKPT)
            print(f'epoch {epoch} done, checkpoint saved')
    return model

if SMOKE:
    print('SMOKE TEST (40 articles) — confirms the loop runs before the full run')
    _m = run_fine_tune(smoke=True)
    print('SMOKE TEST OK — set SMOKE=False and re-run this cell + the next for the full training')

SMOKE TEST (40 articles) — confirms the loop runs before the full run


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SMOKE TEST OK — set SMOKE=False and re-run this cell + the next for the full training


## Fine-tune evaluation

Runs only when `SMOKE=False`. Trains the full model (resumable), then scores it on the test split with
the same five metrics, so the fine-tuned number is directly comparable to the frozen probe above and
to NB5a.

In [9]:
ft_row = None
if not SMOKE:
    model = run_fine_tune(smoke=False)
    model.eval()
    preds, probas = [], []
    with torch.no_grad():
        for i in range(len(val_df)):
            windows = chunk_token_windows(val_df['prep'].iloc[i])
            logit = model(windows, DEVICE, train=False)
            prob = torch.softmax(logit, dim=-1)[0, 1].item()
            probas.append(prob); preds.append(int(prob >= 0.5))
    yte_ft = val_df['label'].to_numpy()
    ft_row = metric_row('AraBERT fine-tuned', yte_ft, np.array(preds), np.array(probas))
    print('TRACK 2 — fine-tuned')
    for k in ['accuracy','precision','recall','macro_f1','auc_roc']:
        print(f'  {k:<10} {100*ft_row[k]:.1f}')
else:
    print('SMOKE mode — skipping full fine-tune evaluation. Set SMOKE=False to run it.')

SMOKE mode — skipping full fine-tune evaluation. Set SMOKE=False to run it.


## Results and comparison

In [10]:
rows = [probe_row] + ([ft_row] if ft_row else [])
res = pd.DataFrame(rows)
show = res.copy()
for c in ['accuracy','precision','recall','macro_f1','auc_roc']:
    show[c] = (100*show[c]).round(1)
print(show.to_string(index=False))

print('\nreference points:')
print('  statistical-only (NB5a, GradientBoosting): 84.7% macro-F1')
print('  cheap-signal control (NB3):                68.8%')
if ft_row:
    gain = 100*(ft_row['macro_f1'] - probe_row['macro_f1'])
    print(f'\nvalue added by fine-tuning: {gain:+.1f} macro-F1 points')

res.to_parquet(f'{OUT_DIR}/nb5b_arabert_results.parquet', index=False)
print('\nsaved nb5b_arabert_results.parquet')

                 model  accuracy  precision  recall  macro_f1  auc_roc
AraBERT frozen + probe      99.5       99.5    99.5      99.5    100.0

reference points:
  statistical-only (NB5a, GradientBoosting): 84.7% macro-F1
  cheap-signal control (NB3):                68.8%

saved nb5b_arabert_results.parquet


## Notes

**What this notebook establishes:**

- Two AraBERT numbers on the pair-aware test split: the frozen linear probe (pre-trained
  representation quality) and the full fine-tune (adapted performance). Their difference is the value
  of fine-tuning.
- Both use the identical sentence-aware chunk-and-pool (overlap of one sentence, cut only at sentence
  boundaries, capped at 6 chunks, mean-pooled). **This exact procedure must be reused in NB5c, NB5d,
  and the hybrid**, or the encoder comparison is not fair.

**For the thesis:**

- Report probe and fine-tune side by side for all three encoders; the probe-vs-fine-tune gap is a
  finding in itself.
- If the fine-tuned AraBERT lands near or below the statistical-only 84.7%, that is an important
  result: it would mean the neural and statistical tracks are complementary rather than the neural
  track dominating, which strengthens the case for fusion. If it lands well above, the hybrid's job is
  to add robustness and interpretability on top.
- State the chunking parameters explicitly (512 limit, sentence-boundary cuts, one-sentence overlap,
  6-chunk cap covering 96% of the corpus, mean pooling).

**Operational:**

- The frozen embeddings are cached to disk, so the probe is cheap to re-run and the cache can be
  uploaded to seed later notebooks.
- Fine-tuning saves a checkpoint every epoch and resumes from it, for Kaggle's 12-hour limit.
- Run the smoke test first (SMOKE=True); only set SMOKE=False once the loop is confirmed.